In [45]:
%reset -f

In [46]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.semi_supervised import LabelSpreading
from sklearn.manifold import SpectralEmbedding
from sklearn.decomposition import PCA
import xgboost as xgb
import catboost as cb
import warnings

warnings.filterwarnings('ignore')

In [47]:
data = pd.read_parquet(Path.cwd().parent.parent / 'TabulatedData' / 'values_all_layers_16bit.parquet')

In [48]:
label = "Prajwal Mundiganal"

# Create Balanced Binary Classification Dataset

In [49]:
# Get data for the specific person
person_data = data[data['name'] == label]
n_person_samples = len(person_data)

print(f"Found {n_person_samples} samples for {label}")

# Get data for all other people
other_data = data[data['name'] != label]

# Sample equal number of data points from other people
# If there are multiple people, sample proportionally or randomly
if len(other_data) >= n_person_samples:
    other_sampled = other_data.sample(n=n_person_samples, random_state=42)
else:
    print(f"Warning: Only {len(other_data)} samples available from other people. Using all available samples.")
    other_sampled = other_data
    # Adjust person samples to match
    person_data = person_data.sample(n=len(other_sampled), random_state=42)
    n_person_samples = len(person_data)

print(f"Using {n_person_samples} samples from {label}")
print(f"Using {len(other_sampled)} samples from other people")

# Combine the datasets
person_data['target'] = 1  # Positive class (the specific person)
other_sampled['target'] = 0  # Negative class (other people)

balanced_data = pd.concat([person_data, other_sampled], ignore_index=True)

# Shuffle the data
balanced_data = balanced_data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset created with {len(balanced_data)} total samples")
print(f"Class distribution: {balanced_data['target'].value_counts().to_dict()}")

Found 169 samples for Prajwal Mundiganal
Using 169 samples from Prajwal Mundiganal
Using 169 samples from other people

Balanced dataset created with 338 total samples
Class distribution: {1: 169, 0: 169}


# Prepare Features and Target

In [50]:
# Features (drop name and target columns)
X = balanced_data.drop(['name', 'target'], axis=1)
y = balanced_data['target']

del balanced_data, data, person_data, other_data, other_sampled
print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")

Feature matrix shape: (338, 25610)
Target distribution: {1: 169, 0: 169}


# Split Data and Scale Features

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
del X, y

Training set: (270, 25610)
Test set: (68, 25610)


# PCA feature reduction

In [52]:
def apply_pca(X_train, X_test, variance_threshold):
    """
    Apply PCA and retain given variance (e.g., 0.99 or 0.95)
    """
    pca = PCA(variance_threshold)

    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    return X_train_pca, X_test_pca, pca

# X_train, X_test, pca = apply_pca(
#         X_train, X_test, 0.8
#     )

# Semi-Supervised Learning Setup

In [53]:
# Prepare spectral embedding and label spreading models separately for better control
def create_spectral_svm_model():
    """Create a pipeline that uses Spectral Embedding + SVM"""

    class SpectralSVMPipeline:
        def __init__(self):
            self.embedder = SpectralEmbedding(
                n_components=50,
                affinity='nearest_neighbors',
                n_neighbors=10,
                random_state=42
            )
            self.svm = SVC(kernel='rbf', C=10, random_state=42)

        def fit(self, X, y):
            X_embed = self.embedder.fit_transform(X)
            self.svm.fit(X_embed, y)
            return self

        def predict(self, X):
            X_embed = self.embedder.fit_transform(X)
            return self.svm.predict(X_embed)

    return SpectralSVMPipeline()


# Define and Train Multiple Models

The algorithms implemented are
- Random Forests
- Support Vector Machine
- Logistic Regression
- Decision Tree
- XGBoost
- Naive Bayes - Gaussian NB
- K-Nearest Neighbours
- Spectral Embedding + Support Vector Machine

In [54]:
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(objective='binary:logistic', random_state=42, tree_method='hist', n_jobs=-1),
    'Naive Bayes-GaussianNB': GaussianNB(),
    'KNN': KNeighborsClassifier(),
    'MLP': MLPClassifier(hidden_layer_sizes=(50,), solver='adam', activation='relu', random_state=42),
    'LabelSpreading': LabelSpreading(kernel='knn', n_neighbors=10),
    # 'Spectral Embedding + SVM': create_spectral_svm_model(),
    "Cat Boost": cb.CatBoostClassifier(random_state=42, thread_count=30, n_estimators=100)
}

results = {}

for name, model in models.items():
    print(f"\n{'=' * 50}")
    print(f"Training {name}...")
    print(f"{'=' * 50}")

    try:
        # Handle semi-supervised models
        if name == 'LabelSpreading':
            # LabelSpreading works directly on features
            model.fit(X_train, y_train)
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)

        elif name == 'Spectral Embedding + SVM':
            # Spectral Embedding for dimensionality reduction
            embedder = SpectralEmbedding(
                n_components=50,
                affinity='nearest_neighbors',
                n_neighbors=10,
                random_state=42
            )

            X_train_embed = embedder.fit_transform(X_train)
            X_test_embed = embedder.fit_transform(X_test)

            # Train SVM on embedded space
            model.fit(X_train_embed, y_train)
            y_pred_train = model.predict(X_train_embed)
            y_pred_test = model.predict(X_test_embed)

        else:
            # Standard supervised models
            model.fit(X_train, y_train)
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)

        # Calculate metrics
        train_acc = accuracy_score(y_train, y_pred_train)
        test_acc = accuracy_score(y_test, y_pred_test)

        results[name] = {
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'model': model
        }

        print(f"{name} Results:")
        print(f"Training Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")

        # Print classification report
        print("\nClassification Report (Test Set):")
        print(classification_report(y_test, y_pred_test, target_names=[f'Not {label}', label]))

    except Exception as e:
        print(f"Error training {name}: {str(e)}")
        results[name] = {'error': str(e)}



Training Random Forest...
Random Forest Results:
Training Accuracy: 1.0000
Test Accuracy: 0.9559

Classification Report (Test Set):
                        precision    recall  f1-score   support

Not Prajwal Mundiganal       0.94      0.97      0.96        34
    Prajwal Mundiganal       0.97      0.94      0.96        34

              accuracy                           0.96        68
             macro avg       0.96      0.96      0.96        68
          weighted avg       0.96      0.96      0.96        68


Training SVM...
SVM Results:
Training Accuracy: 0.9926
Test Accuracy: 0.8529

Classification Report (Test Set):
                        precision    recall  f1-score   support

Not Prajwal Mundiganal       0.77      1.00      0.87        34
    Prajwal Mundiganal       1.00      0.71      0.83        34

              accuracy                           0.85        68
             macro avg       0.89      0.85      0.85        68
          weighted avg       0.89      0.85  

# Summary Results

In [55]:
print("\n" + "=" * 80)
print("SUMMARY OF RESULTS")
print("=" * 80)

summary_data = []
for name, result in results.items():
    if 'error' not in result:
        summary_data.append({
            'Model': name,
            'Train Accuracy': f"{result['train_accuracy']:.4f}",
            'Test Accuracy': f"{result['test_accuracy']:.4f}"
        })
    else:
        summary_data.append({
            'Model': name,
            'Train Accuracy': 'Error',
            'Test Accuracy': result['error']
        })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Find best performing model
valid_results = {k: v for k, v in results.items() if 'error' not in v}
if valid_results:
    best_model = max(valid_results.items(), key=lambda x: x[1]['test_accuracy'])
    print(f"\n{'=' * 50}")
    print(f"Best Performing Model: {best_model[0]}")
    print(f"Test Accuracy: {best_model[1]['test_accuracy']:.4f}")
    print(f"{'=' * 50}")


SUMMARY OF RESULTS
                 Model Train Accuracy Test Accuracy
         Random Forest         1.0000        0.9559
                   SVM         0.9926        0.8529
   Logistic Regression         0.9963        0.9559
         Decision Tree         1.0000        0.9265
               XGBoost         1.0000        0.9559
Naive Bayes-GaussianNB         0.9889        0.7353
                   KNN         0.9111        0.8235
                   MLP         0.9926        0.9412
        LabelSpreading         0.8296        0.8676

Best Performing Model: Random Forest
Test Accuracy: 0.9559


# Notes
1. Need to take samples from everyone